In [1]:
import random
import pandas as pd
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

## 1. Load and Shuffle Data

In [3]:
nltk.download('movie_reviews', quiet=True)
documents = [(" ".join(movie_reviews.words(fileid)), category) 
             for category in movie_reviews.categories() 
             for fileid in movie_reviews.fileids(category)]

random.seed(42)
random.shuffle(documents)

X = [doc[0] for doc in documents]
y = [1 if doc[1] == 'pos' else 0 for doc in documents]

In [4]:
X[0]

"mr . bean , a bumbling security guard from england is sent to la to help with the grandiose homecoming of a masterpiece american painting . the first two words should have said enough to let you know what occurs during bean ' s trip to la , but if they didn ' t look out because you are in for a rather interesting if not odd ride . heck depending on your humor you might end up laughing through the whole flick . either way look out america bean is coming . well , what can really be said about this movie , there is very little discernible plot . that much is not hard to grapple with for it is a slapstick comedy . it achieves that goal rather admirably , but because it is that , the plot is just screaming for help . the whole premise that the movie is based on is to say the least flawed . the movie had its funny moments but there was no real story line other than something that could be thought up on a whim and carried through and in many causes ad - libbed as you went . don ' t go into t

## 2. Train-Test Split (80% Train, 20% Test)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

## 3. TF-IDF Feature Extraction

In [6]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_test_vec = vectorizer.transform(X_test)

In [7]:
X_train_vec

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.04248375, 0.21012396, 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(1600, 5000))

## 4. Define and Evaluate Algorithms

In [8]:
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(),
    "LinearSVC": LinearSVC(C=0.5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

In [9]:
results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    fitted_models[name] = model
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })

## 5. Display Model Comparison Table

In [10]:
print("=== MODEL COMPARISON RESULTS ===")
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

=== MODEL COMPARISON RESULTS ===
                  Model  Accuracy  Precision  Recall  F1 Score
Multinomial Naive Bayes    0.7850   0.816667   0.735  0.773684
    Logistic Regression    0.8100   0.800971   0.825  0.812808
              LinearSVC    0.8475   0.832536   0.870  0.850856
          Random Forest    0.7600   0.820988   0.665  0.734807


## 6. Detailed Classification Report for Best Model (LinearSVC)

In [11]:
best_model = fitted_models["LinearSVC"]
y_pred_best = best_model.predict(X_test_vec)

print("\n=== DETAILED REPORT FOR BEST MODEL (LinearSVC) ===")
print(classification_report(y_test, y_pred_best, target_names=["Negative", "Positive"]))


=== DETAILED REPORT FOR BEST MODEL (LinearSVC) ===
              precision    recall  f1-score   support

    Negative       0.86      0.82      0.84       200
    Positive       0.83      0.87      0.85       200

    accuracy                           0.85       400
   macro avg       0.85      0.85      0.85       400
weighted avg       0.85      0.85      0.85       400



## 7. Demo / Test on Custom User Sentences

In [12]:
print("=== CUSTOM DEMO PREDICTIONS ===")
sample_reviews = [
    "An absolute cinematic masterpiece with breathtaking visual performance!",
    "Boring plot, terrible direction, and completely predictable script."
]
sample_vec = vectorizer.transform(sample_reviews)
sample_preds = best_model.predict(sample_vec)

for text, pred in zip(sample_reviews, sample_preds):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"Review: '{text}' --> Sentiment: {sentiment}")

=== CUSTOM DEMO PREDICTIONS ===
Review: 'An absolute cinematic masterpiece with breathtaking visual performance!' --> Sentiment: Positive
Review: 'Boring plot, terrible direction, and completely predictable script.' --> Sentiment: Negative


In [13]:
pip install streamlit joblib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import joblib

# Save vectorizer and model
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(best_model, 'sentiment_model.pkl')

print("Model and Vectorizer saved successfully!")

Model and Vectorizer saved successfully!


In [15]:
%%writefile app.py
import streamlit as st
import joblib

# Set page configuration
st.set_page_config(
    page_title="Movie Review Sentiment Analyzer",
    page_icon="🎬",
    layout="centered"
)

# Load saved vectorizer and model with caching
@st.cache_resource
def load_artifacts():
    vectorizer = joblib.load('tfidf_vectorizer.pkl')
    model = joblib.load('sentiment_model.pkl')
    return vectorizer, model

try:
    vectorizer, model = load_artifacts()
except Exception as e:
    st.error("Error loading model files. Make sure 'tfidf_vectorizer.pkl' and 'sentiment_model.pkl' exist.")
    st.stop()

# App Title & Header
st.title("🎬 Movie Review Sentiment Analyzer")
st.markdown("Enter a movie review below to classify its sentiment as **Positive** or **Negative**.")

# User Input Text Area
review_text = st.text_area(
    "Movie Review:",
    height=150,
    placeholder="e.g., An absolute cinematic masterpiece with breathtaking visual performance!"
)

# Predict Button
if st.button("Analyze Sentiment", type="primary"):
    if not review_text.strip():
        st.warning("Please enter a review before analyzing.")
    else:
        transformed_text = vectorizer.transform([review_text])
        prediction = model.predict(transformed_text)[0]

        st.markdown("---")
        st.subheader("Prediction Result")

        if prediction == 1:
            st.success("🟢 **Positive Sentiment**")
        else:
            st.error("🔴 **Negative Sentiment**")

Overwriting app.py


In [ ]:
!streamlit run app.py